# 03 Exploratory Data Analysis

Creates numerical, categorical, missing-value, correlation, boxplot, and distribution plots where supported. Plots are saved to `ai/preprocessing/eda`.


In [ ]:
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    import seaborn as sns
    sns.set_theme(style='whitegrid')
except Exception:
    sns = None

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'ai').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
DATA_DIR = PROJECT_ROOT / 'ai' / 'datasets'
PREPROCESSING_DIR = PROJECT_ROOT / 'ai' / 'preprocessing'
MODEL_DIR = PROJECT_ROOT / 'ai' / 'models'

CSV_TSV_EXTENSIONS = {'.csv', '.tsv'}

def discover_data_files(data_dir=DATA_DIR):
    return sorted(p for p in data_dir.rglob('*') if p.suffix.lower() in CSV_TSV_EXTENSIONS)

def read_dataset(path):
    sep = '\t' if path.suffix.lower() == '.tsv' else ','
    return pd.read_csv(path, sep=sep)

def safe_name(path):
    return path.relative_to(DATA_DIR).as_posix().replace('/', '__')

def display_heading(text):
    try:
        from IPython.display import display, Markdown
        display(Markdown(f'### {text}'))
    except Exception:
        print(f'\n### {text}')

EDA_DIR = PREPROCESSING_DIR / 'eda'
EDA_DIR.mkdir(parents=True, exist_ok=True)

def save_current_plot(out_dir, name):
    out_dir.mkdir(parents=True, exist_ok=True)
    path = out_dir / f'{name}.png'
    plt.tight_layout()
    plt.savefig(path, dpi=150, bbox_inches='tight')
    plt.show()
    plt.close()
    return path

def plot_missing(df, out_dir):
    missing = df.isna().sum()
    if missing.sum() == 0:
        print('No missing values to plot.')
        return
    missing[missing > 0].sort_values().plot(kind='barh', figsize=(8, max(3, len(missing[missing>0]) * 0.35)))
    plt.title('Missing values by column')
    save_current_plot(out_dir, 'missing_values')


In [ ]:
for path in discover_data_files():
    display_heading(path.relative_to(DATA_DIR).as_posix())
    try:
        df = read_dataset(path)
        out_dir = EDA_DIR / safe_name(path).replace(path.suffix, '')
        numeric = df.select_dtypes(include='number')
        categorical = df.select_dtypes(exclude='number')
        if not numeric.empty:
            numeric.hist(figsize=(12, 8), bins=25)
            save_current_plot(out_dir, 'histograms')
            if numeric.shape[1] >= 2:
                plt.figure(figsize=(10, 7))
                corr = numeric.corr(numeric_only=True)
                if sns:
                    sns.heatmap(corr, annot=False, cmap='coolwarm', center=0)
                else:
                    plt.imshow(corr, cmap='coolwarm')
                    plt.xticks(range(len(corr)), corr.columns, rotation=90)
                    plt.yticks(range(len(corr)), corr.columns)
                    plt.colorbar()
                plt.title('Correlation heatmap')
                save_current_plot(out_dir, 'correlation_heatmap')
            for col in numeric.columns[:12]:
                plt.figure(figsize=(7, 4))
                if sns:
                    sns.boxplot(x=df[col])
                else:
                    plt.boxplot(df[col].dropna(), vert=False)
                plt.title(f'Boxplot: {col}')
                save_current_plot(out_dir, f'boxplot_{col}')
                plt.figure(figsize=(7, 4))
                if sns:
                    sns.histplot(df[col].dropna(), kde=True)
                else:
                    df[col].dropna().plot(kind='hist', bins=25)
                plt.title(f'Distribution: {col}')
                save_current_plot(out_dir, f'distribution_{col}')
        else:
            print('No numeric columns found.')
        plot_missing(df, out_dir)
        for col in categorical.columns[:12]:
            counts = df[col].astype(str).value_counts().head(20)
            if counts.empty:
                continue
            plt.figure(figsize=(9, 5))
            counts.sort_values().plot(kind='barh')
            plt.title(f'Top values: {col}')
            save_current_plot(out_dir, f'value_counts_{col}')
    except Exception as exc:
        print(f'EDA failed for {path.name}: {exc}')
